# Overview
This jupyter notebook constructs a fictional dataset in BIDS format using the "muniverse" ( https://github.com/dfarinagroup/muniverse/ ) framework. 

The experimental setup is intentionally more complex than most regular datasets for illustrative purposes. 

The dataset contains 6 subjects who each performed 3 tasks. 
There are 3 Grids of HDsEMG. There are 2 Grids of HDiEMG (thin filaments). Additionally there are 2 Fine-Wire Electrodes and 2 Needle Electrodes, as well as 2 reference electrodes and 1 ground electrode. Grids have different numbers of rows and columns. 

<img src="InputMetadata/experimentalSetup.jpg" alt="Drawing" style="width: 600px;"/> <img src="InputMetadata/schematics.jpg" alt="Drawing" style="width: 400px;"/>







In [ ]:
import numpy as np
import pandas as pd
import muniverse.utils.bids_routines
import copy

# Metadata 
## Global (dataset wide) metadata
### Subjects data & sidecar
We define the data of our subjects. This will end up in the file "participants.tsv". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/data-summary-files.html#participants-file 

In [ ]:
subjects_data = {
    "participant_id": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06"],
    "age": [42, 43, 44, 45, 46, 47],
    "sex": ["M", "F", "M", "F", "M", "F"],
    "handedness": ["R", "R", "R", "R", "L", "R"],
    "weight": [70, 68, 66, 64, 62, 60],
    "height": [1.7, 1.72, 1.74, 1.76, 1.78, 1.8],
    "group": ["T", "T", "T", "P", "P", "P"],
}

We define the subjects sidecar file. This information will end up in "participants.json". 
We only need to do this for extra columns, since the following defaults are already added by muniverse: "participant_id", "age", "sex", "handedness", "weight", "height". 

In [ ]:
subjects_sidecar = {
    "group": {
        "Description": "Group the subject belongs to.",
        "Levels": {
            "T": "Treatment", 
            "P": "Placebo"
        }
    }
}

### Dataset sidecar 
We define the dataset sidecar. This will end up in "dataset_description.json". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/dataset-description.html#dataset_descriptionjson 

In [ ]:
dataset_sidecar = {
  "Name": "DatasetName",
  "License": "CC BY 4.0",
  "Authors": [
    "alice",
    "bob"
  ], # has to be a list even if it has only one entry 
  "ReferencesAndLinks": [
    "citation of related publication as text",
    "related publication as DOI"
  ], # has to be a list even if it has only one entry 
  "EthicsApprovals": [
    "number of ethics approval."
  ], # has to be a list even if it has only one entry 
  "GeneratedBy": [{"Name":"munitquest tutorials"}], # must be array of objects 
  
}

### Readme File 
We define a readme for the dataset. This becomes "readme.md". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/dataset-description.html#readme 
A readme Template can be found here: https://bids.neuroimaging.io/getting_started/templates/index.html?h=readme#readmemd 


In [ ]:
readme = """
Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.

Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
"""

### Tasks
We define the names of our tasks. 

In [ ]:
tasklist = ["baselinenoise", "trapezoidalContraction30PercentMVC", "trapezoidalContraction50PercentMVC"]

## Local (recording specific) metadata
### EMG sidecar 
We define the emg sidecar. This will end up in files ending with "..._emg.json". There will be one such file per recording. 
In our case they are identical for each recording, except for task related information such as *Taskname* and *TaskDescription* and *Instructions*. 
More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#sidecar-json-_emgjson 

In [ ]:
emg_sidecar_task1 = {
    "EMGPlacementScheme": "Measured", 
    "EMGPlacementSchemeDescription": "(i) Surface EMG: Four grids were carefully positioned side-to-side with a 4-mm distancebetween the electrodes at the edges of adjacent grids. The 256 electrodes were centered to the muscle belly (right tibialis anterior) and laid  within the muscle perimeter identified through palpation. (ii) Invasive thin film EMG: A guiding filament inserted into a 25-gauge needle was used to place the thin-film structure in the muscle. Afterwards, the needle is withdrawn, leaving only the thin-film array within the muscle. (iii) Invasive Fine Wire EMG: The fine-wires were inserted as pairs using a cannulae to a target depth of 2 cm. After the insertion, the bipolar EMG was visually inspected and if needed the electrode positions were adjusted through light needle movements. Afterwards, the cannula was removed leaving only the fine wires within the muscle. (iv) Invasive Needle EMG: The needle was inserted (perpendicular to the skin) into the proximal part of the ADM targeting an insertion depth of 1 cm. ",
    "EMGReference": "ChannelSpecific",
    "EMGGround": "G1",
    "SamplingFrequency": 2048,
    "PowerLineFrequency": 50,
    "RecordingType": "continuous", 
    "HardwareFilters": {
        "Anti-aliasing filter": {
            "half-amplitude cutoff (Hz)": 500,
            "Roll-off": "6dB/Octave"
        },
        "some other filter": {
            "filterparameter 1": 500,
            "filterparameter 2": "12dB/Octave",
            "filterparameter 3": 80
        }
    }, 
    "SoftwareFilters": "n/a",
    "EMGChannelCount":45,
    "TaskName": "baselinenoise",
    "TaskDescription": "Relaxed muscle for 5 seconds.",
    "Instructions": "Relax your muscle completely.", 
    "Preamplification": 1,
    "Gain": 1,
    "Manufacturer": "some amplifier manufacturer",
    "ManufacturersModelName": "some amplifier model name"
}

# Since almost everything is the same for the other tasks we create a copy and only change task related fields. 
emg_sidecar_task2 = copy.deepcopy(emg_sidecar_task1)
emg_sidecar_task2["TaskName"] = "trapezoidalContraction30PercentMVC"
emg_sidecar_task2["TaskDescription"] = "A trapezoidal contraction at 30 percent MVC, consisting of linear ramps up and down performed at 30 percent per second and a plateau maintained for 1 s."
emg_sidecar_task2["Instructions"] = "Follow path provided via visual feedback."

emg_sidecar_task3 = copy.deepcopy(emg_sidecar_task1)
emg_sidecar_task3["TaskName"] = "trapezoidalContraction50PercentMVC"
emg_sidecar_task3["TaskDescription"] = "A trapezoidal contraction at 50 percent MVC, consisting of linear ramps up and down performed at 50 percent per second and a plateau maintained for 1 s."
emg_sidecar_task3["Instructions"] = "Follow path provided via visual feedback."

# To retrieve these cleanly later from inside a for-loop we pack them into a dict with tasknames as keys 
emg_sidecar_taskDict = {
    "baselinenoise": emg_sidecar_task1, 
    "trapezoidalContraction30PercentMVC": emg_sidecar_task2, 
    "trapezoidalContraction50PercentMVC": emg_sidecar_task3, 
}

### Coordinate system sidecar 

Next we define the coordinate systems our electrodes will be positioned inside. For grids of electrodes the recommended approach is to define electrode positions in a local grid specific coordinate systems, and then locate the grid specific coordinate system inside an anatomical parent coordinate system. 

Our parent coordinate system is defined through anatomical landmarks on the forearm, with percent between landmarks as the coordinate unit. The grid specific systems are then located by designating an anchor electrode and its location in the parent coordinate system. 

More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#coordinate-system-json-_coordsystemjson 

In [ ]:
coord_sidecar = {
    "forearmCoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "percent",
        "EMGCoordinateSystemDescription": "x: Radial Styloid Process (RSP) -> Ulnar Styloid Process (USP); y: Right-hand rule (limits: Olecranon Process -> Cubital Fossa); z: midpoint RSP-USP -> Lateral Humerus Epicondyle (LHE)"
    }, 

    "grid1CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": " The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E001",
        "AnchorCoordinates": [30, 50, 80]
    }, 

    "grid2CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E009",
        "AnchorCoordinates": [30, 50, 60]
    }, 

    "grid3CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E017",
        "AnchorCoordinates": [30, 50, 40]
    }, 

    "intraGrid1CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "iE001",
        "AnchorCoordinates": [30, 50, 70]
    }, 

    "intraGrid2CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "iE009",
        "AnchorCoordinates": [30, 50, 50]
    }
}

### Electrode data & sidecar
We load electrode metadata from a file. This becomes files called "..._electrodes.tsv". Instead of loading from a file you could also write code that builds the desired dataframe. More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#electrodes-description-_electrodestsv 


In [ ]:
el_metadata = pd.read_excel("InputMetadata//electrodes.xlsx")
el_metadata.head()

We define the electrodes sidecar. This becomes "..._electrodes.json" files. Similar to the subjects sidecar we only need to define the non-default columns. 

In [ ]:
electrodes_sidecar = {
    "interelectrodeDistance": "Distance between electrodes. In a grid this means distance between neighboring electrodes. In a fine wire it means distance between the wire tips.", 
    "electrodeSurfaceArea": "Surface area of the electrode", 
    "fineWireDiameter": "Diameter of the fine wire tip", 
    "fineWireRecordingTipLength": "Unisolated length of the fine wire tip", 
    "concentricNeedleDiameter": "Concentric needle diameter", 
    "concentricNeedleLength": "Length of concentric needle", 
    "ElectrodeManufacturer": "Name of electrode manufacturer", 
    "ElectrodeManufacturersModelName": "Model name of Electrode", 
    "electrodeType": "Type of electrode", 
}

### Channel data & sidecar
We load channel data from a file. This becomes "..._channels.tsv". More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#channels-description-_channelstsv 

In [ ]:
ch_metadata = pd.read_excel("InputMetadata//channels.xlsx")
ch_metadata.head()

We define channels sidecar. This becomes "..._channels.json" files. Again we only need to define non-default columns. 

In [ ]:
channels_sidecar = {
    "someAdditionalColumn": "some Description of this column"
}

### Events data & sidecar
We load event data from a file. This becomes "..._events.tsv". 

For more information see: https://bids-specification.readthedocs.io/en/stable/modality-agnostic-files/events.html 

In [ ]:
events_metadata1 = pd.read_csv("InputMetadata//events30PercentMVC.tsv", sep="\t") # TODO baseline events ???? 
events_metadata2 = pd.read_csv("InputMetadata//events30PercentMVC.tsv", sep="\t")
events_metadata3 = pd.read_csv("InputMetadata//events50PercentMVC.tsv", sep="\t")
# To retrieve these cleanly later from inside a for-loop we pack them into a dict with tasknames as keys 
events_metadata_taskDict = {
    "baselinenoise": events_metadata1, 
    "trapezoidalContraction30PercentMVC": events_metadata2, 
    "trapezoidalContraction50PercentMVC": events_metadata3,    
}
events_metadata1.head()

We define events sidecar. This becomes "events.json". Again we need to define non-default columns. 

In [ ]:
events_sidecar = {
    "sample": {
        "Description": "Sample index of the event onset (zero-indexing)",
        "Unit": "samples"
    },
    "mvc_rate": {
        "Description": "Rate at which the torque changes in percent MVC per second",
        "Unit": "% MVC / s"
    },
    "mvc_level": {
        "Description": "MVC (maximum voluntary contraction) level at the onset of the event",
        "Unit": "% MVC"
    },
    "event_type": {
        "Description": "Event label.",
        "Levels": {
            "muscle_on": "The muscle is activated.",
            "muscle_off": "The muscle is deactivated.",
            "linear_isometric_ramp": "The isometric torque changes linearly over time with a fixed rate.",
            "steady_isometric": "Steady isometric contraction at a fixed MVC level."
        }
    },
    "description": {
        "Description": "Free text event description."
    }
}

# Building the dataset
Now we create an instance of a muniverse bids_dataset. 
First we set global metadata. 

In [ ]:
FictionalDatasetExample = muniverse.utils.bids_routines.bids_dataset(datasetname="FictionalDatasetExample")
FictionalDatasetExample.set_metadata(field_name='subjects_data', source=subjects_data)
FictionalDatasetExample.set_metadata(field_name='subjects_sidecar', source=subjects_sidecar)
FictionalDatasetExample.set_metadata(field_name='dataset_sidecar', source=dataset_sidecar)
FictionalDatasetExample.readme = readme
FictionalDatasetExample.write()

Then we loop over all participants and tasks, create an instance of bids_emg_recording for each, add all local metadata and the raw data to it. In this case we randomly generate the raw data, but you could easily replace that with a function like *get_raw_data(participant_id, task)* to insert your own data. 

Some metadata is the same for all tasks within a subject or even for all subjects. Instead of writing a redundant file for each subject and each task we can use inheritance to save some discspace. This is done with the arguments *inherited_metadata* and *inherited_level* that are passed to the *bids_emg_recording* class. 

For more information about inheritance see: https://bids-specification.readthedocs.io/en/stable/common-principles.html#the-inheritance-principle 

In [ ]:
rng = np.random.default_rng(seed=12345)

# manual override to write only one subject and one task: 
subjects_data["participant_id"] = ["sub-01"]
tasklist = ["baselinenoise"]

for participant_id in subjects_data["participant_id"]:
    participant_id = participant_id[4:]
    print(f"Bidsifying data of sub-{participant_id}")

    for task in tasklist:
        # create a random array of the correct size to be our raw data. 
        n_channels = len(ch_metadata.loc[:,"name"])
        recordingLength = 5 # seconds 
        samplingFrequency = 2048 # samples per second 
        n_samples = np.ceil(recordingLength * samplingFrequency)
        data = rng.uniform(low=0, high=1, size=(n_channels, n_samples))

        # Make a recording and add data and metadata
        emg_recording = muniverse.utils.bids_routines.bids_emg_recording(
            parent_dataset = FictionalDatasetExample, 
            subject_label=participant_id, 
            task_label=task, 
            datatype='emg',
            inherited_metadata=["coordsystem.json", "electrodes.tsv", "electrodes.json", "events.json"], # Here is where we define which files to inherit
            inherited_level=["subject", "subject", "dataset", "dataset"], # and here which level they are inherited to 
        )
        emg_recording.set_metadata(field_name='emg_sidecar', source=emg_sidecar_taskDict[task])
        emg_recording.set_data(field_name='emg_data', mydata=data,fsamp=samplingFrequency)
        emg_recording.set_metadata(field_name='coord_sidecar', source=coord_sidecar, overwrite=True)
        emg_recording.set_metadata(field_name='electrodes_sidecar', source=electrodes_sidecar)
        emg_recording.set_metadata(field_name='electrodes', source=el_metadata) 
        emg_recording.set_metadata(field_name='channels_sidecar', source=channels_sidecar)
        emg_recording.set_metadata(field_name='channels', source=ch_metadata)

        emg_recording.set_metadata(field_name="events_sidecar", source=events_sidecar)
        emg_recording.set_metadata(field_name="events", source=events_metadata_taskDict[task])
        emg_recording.write()


# Validation
Finally we use the BIDS-validator to check the integrity of our dataset. 
For more information see here: https://github.com/bids-standard/bids-validator 

In [ ]:
err, warn, _ = FictionalDatasetExample.validate(
    print_errors=True,
    print_warnings=True,
    ignored_codes=[],
    ignored_fields=[ # We ignore some fields that would produce warnings
        "SourceDatasets", # there isn't one
        "DeviceSerialNumber", # no hardware was used to record this dataset
        "SoftwareVersions", # this also pertains to measurement hardware 
        "InstitutionName", # we also don't want to specify an affiliated institution 
        "InstitutionAddress",
        "InstitutionalDepartmentName",

        "HEDVersion", # we don't use HED. 
        "StimulusPresentation", # we also ignore this, to not clutter this jupyter notebook. 

        # We have different ElectrodeManufacturer and ElectrodeManufacturersModelName per electrode, so we specify it on a per electrode basis in the electrodes.tsv file, rather than inside of emg.json (as mandated by the BIDS documentation for this case). However at the time of writing this script, the Validator is not smart enough to understand this. So we ignore the raised warning. 
        "ElectrodeManufacturer", 
        "ElectrodeManufacturersModelName", 
        
    ],
    ignored_files=[],
)

print("The BIDS conversion has completed")
print(f"Your BIDS dataset contains {len(err)} errors and {len(warn)} warnings")